# **Notebook 4: RAG Implementation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] `outputs.json` — Created by Notebook 3
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `./chroma_db/` — Persisted ChromaDB vector index _(Required by NB5 and NB7)_
- [ ] `outputs.json` (updated) — adds `naive_rag_output` _(Required by NB5 and NB7)_

---

### **Task 3.2: Implement Retrieval-Assisted Generation**

#### **3.2.1 Generate Embeddings [4 marks]**
**The Task:** Initialise the `all-MiniLM-L6-v2` embedding model, embed the SOP documents, and validate the embeddings were produced.

**Hints & Tips:**
* Load the SOP documents with `TextLoader` first (or reuse the corpus from NB2).
* `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")` runs on CPU — no GPU needed.
* Validate by embedding one sample string and checking the vector length (384 dims for MiniLM).
* You MUST use the same embedding model when reloading in Notebooks 5 and 7.

**Embedding Model Options:**
* **`all-MiniLM-L6-v2`** (recommended): 384-dim, fast, ~80MB.
* **`all-mpnet-base-v2`**: 768-dim, higher quality, slower.
* **`bge-small-en-v1.5`**: 384-dim, newer architecture.

**Learner Inference:** Your text is now coordinates in semantic space — similar meanings sit close together.

In [1]:
# YOUR CODE HERE
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import TextLoader
import glob
import os

EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading embedding model: {EMBEDDING_MODEL_ID}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_ID)

# Validate embedding vector shape
sample_vec = embedding_model.encode("Where is my package?")
print(f"Sample embedding shape: {sample_vec.shape} (Expected: (384,))")
assert sample_vec.shape[0] == 384, "Embedding dimensions mismatch!"

sop_files = sorted(glob.glob("corporate_policies/*.md") or glob.glob("Dataset/Dataset/sop_documents/*.md"))
sop_docs = []
for fpath in sop_files:
    loader = TextLoader(fpath, encoding='utf-8')
    docs = loader.load()
    for d in docs:
        d.metadata["source"] = os.path.basename(fpath)
        d.metadata["doc_id"] = os.path.basename(fpath).replace(".md", "")
        sop_docs.append(d)

print(f"Loaded {len(sop_docs)} SOP documents.")

/tmp/ipykernel_35126/3197996336.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sample embedding shape: (384,) (Expected: (384,))
Loaded 13 SOP documents.


#### **3.2.2 Build Vector Index [4 marks]**
**The Task:** Create a persistent Chroma vector index from the embedded documents, configure similarity search, and validate the index.

**Hints & Tips:**
* `Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")` auto-saves — no manual `.persist()` needed.
* Validate with `vector_db._collection.count()` — should equal the number of SOP documents.
* Run one test `.similarity_search("refund", k=1)` to confirm retrieval works.

**Vector DB Options:**
* **ChromaDB** (recommended): simple API, auto-persistence, LangChain integration.
* **FAISS**: faster for >100K docs, but no built-in persistence (manual serialization).

**Learner Inference:** The index lets you search by meaning — it returns the document mathematically closest to your query's coordinates.

In [2]:
# YOUR CODE HERE
import chromadb

persist_dir = "./chroma_db"
chroma_client = chromadb.PersistentClient(path=persist_dir)

collection_name = "corporate_sops"
try:
    chroma_client.delete_collection(collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)

doc_ids = [d.metadata["doc_id"] for d in sop_docs]
doc_texts = [d.page_content for d in sop_docs]
doc_metas = [d.metadata for d in sop_docs]
doc_embeddings = embedding_model.encode(doc_texts).tolist()

collection.add(
    ids=doc_ids,
    documents=doc_texts,
    metadatas=doc_metas,
    embeddings=doc_embeddings
)

print(f"ChromaDB Persistent Collection '{collection_name}' created at {persist_dir}")
print(f"Indexed documents count: {collection.count()}")

res = collection.query(
    query_embeddings=embedding_model.encode(["refund request policy"]).tolist(),
    n_results=1
)
print(f"Test query top retrieval: {res['ids'][0][0]} (Cosine Distance: {res['distances'][0][0]:.4f})")

ChromaDB Persistent Collection 'corporate_sops' created at ./chroma_db
Indexed documents count: 13
Test query top retrieval: refund_policy (Cosine Distance: 0.4276)


#### **3.2.3 Implement Retrieval Workflow [4 marks]**
**The Task:** Execute a "Naive RAG" workflow — pass the raw customer query into the vector DB, fetch the top result, and augment the LLM prompt.

**Hints & Tips:**
* Use `.similarity_search(query, k=1)` for the top-1 document.
* Check whether the raw query retrieved the WRONG policy — common with ambiguous queries.
* Inject context via the system prompt: `"Answer strictly using this SOP: {context}"`.

**Parameter Tuning:**
* `k=1`: one document (focused). `k=3`: more context if SOPs overlap. `k=5`: max, risks long prompts.

**Learner Inference:** Noisy queries often retrieve the wrong document — proving Naive RAG is flawed and motivating the fine-tuned router.

In [3]:
# YOUR CODE HERE
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

with open("outputs.json", "r", encoding="utf-8") as f:
    outputs_data = json.load(f)

test_query = outputs_data["test_query"]

# Naive RAG retrieval using raw customer query
query_embedding = embedding_model.encode([test_query]).tolist()
retrieval_res = collection.query(
    query_embeddings=query_embedding,
    n_results=1
)

retrieved_doc_id = retrieval_res['ids'][0][0]
retrieved_context = retrieval_res['documents'][0][0]
print(f"Query: {test_query}")
print(f"Retrieved Document: {retrieved_doc_id}")
print(f"Context snippet: {retrieved_context[:200]}...")

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)
model.eval()

system_prompt = "You are a customer service assistant. Answer the customer query strictly using ONLY the following SOP context:\n\n" + retrieved_context
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_query}
]

rag_prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(rag_prompt_text, return_tensors="pt").to(device)

with torch.no_grad():
    out_ids = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.pad_token_id
    )

in_len = inputs.input_ids.shape[1]
naive_rag_output = tokenizer.decode(out_ids[0][in_len:], skip_special_tokens=True).strip()

print(f"\nNaive RAG Output:\n{naive_rag_output}")

Query: Where is my package? I ordered it last week and it is still delayed.
Retrieved Document: shipping_delays
Context snippet: # Shipping Delays

## Scope
This procedure covers orders that have shipped but are running behind the
estimated delivery date, as well as orders stuck in a pre-shipment state.

## Standard Timelines
D...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Naive RAG Output:
I'm sorry to hear that your package is still delayed. To help you better, could you please provide me with the order number and the original estimated delivery date when you placed your order? This will allow me to check the current status and assist you further.


---
## Save Artifacts for Downstream Notebooks

In [4]:
# YOUR CODE HERE
outputs_data["naive_rag_output"] = naive_rag_output
outputs_data["naive_rag_retrieved_doc"] = retrieved_doc_id

with open("outputs.json", "w", encoding="utf-8") as f:
    json.dump(outputs_data, f, indent=4)

print("Updated outputs.json with naive_rag_output.")

Updated outputs.json with naive_rag_output.


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 5.**

- [ ] SOP documents loaded via `TextLoader`
- [ ] **Embeddings generated and validated** ← _Task 3.2.1_
- [ ] **ChromaDB index built, validated, and persisted** ← _Task 3.2.2_
- [ ] Naive similarity search executed on `test_query`
- [ ] Naive RAG output generated with SOP context injected
- [ ] **`./chroma_db/` exists on disk** ← _CRITICAL for NB5 and NB7_
- [ ] **`outputs.json` updated** with `naive_rag_output` ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**